In [0]:
%sql

CREATE OR REPLACE TABLE traffic_catalog.gold.gold_congestion_by_road
USING DELTA
AS

SELECT
    road_id,
    road_name,
    road_code,

    COUNT(*) AS total_measurements,

    SUM(
        CASE
            WHEN current_speed / NULLIF(free_flow_speed, 0) < 0.70
                THEN 1
            ELSE 0
        END
    ) AS congested_measurements,

    SUM(
        CASE
            WHEN current_speed / NULLIF(free_flow_speed, 0) >= 0.70
             AND current_speed / NULLIF(free_flow_speed, 0) < 0.85
                THEN 1
            ELSE 0
        END
    ) AS moderate_measurements,

    SUM(
        CASE
            WHEN current_speed / NULLIF(free_flow_speed, 0) >= 0.85
                THEN 1
            ELSE 0
        END
    ) AS normal_measurements,

    ROUND(
        SUM(
            CASE
                WHEN current_speed / NULLIF(free_flow_speed, 0) < 0.70
                    THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS congestion_percentage

FROM traffic_catalog.silver.silver_traffic

GROUP BY
    road_id,
    road_name,
    road_code;